In [8]:
from fonctions import *
import cv2,os,numpy as np


In [34]:
x = []
y = []
for classe in ["cats","dogs"]:
    dossier = os.path.join("datasets/train",classe)
    for fichier in tqdm(os.listdir(dossier)):
        chemin = os.path.join(dossier,fichier)
        image = cv2.imread(chemin)
        image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
        image = cv2.resize(image,(64,64))
        ## Normalisation
        image = image/255.0
        ## flatten
        ## mes images sont de tailles 64x64x3
        x.append(image)

        if classe == "cats":
            y.append(0)
        else:
            y.append(1)

x = np.array(x)
y = np.array(y)


100%|██████████| 12500/12500 [00:42<00:00, 293.00it/s]


In [ ]:
print(x.shape)
## Le shape de x correspond a 25000 images de chat et des chiens, de dimensions 64*64*3
print(y.shape)
## y represnte les classes, "0" pour chat et "1" pour chien

(25000, 64, 64, 3)
(25000,)


In [36]:
X_train,X_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


X_train: (20000, 64, 64, 3)
X_test: (5000, 64, 64, 3)
y_train: (20000,)
y_test: (5000,)


In [ ]:
## je les convertis en float 32 pour qu'elles prennent moins de mémoire lors de l'exécution

X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

In [39]:
## j'applatis mes données pour le reseau de neurones
## 12288 = 64*64*3
## Je veux partir en format(features,exemples)
X_train_flat,X_test_flat = X_train.reshape(12288,X_train.shape[0]),X_test.reshape(12288,X_test.shape[0])
print(X_train_flat.shape)
print(X_test_flat.shape)

(12288, 20000)
(12288, 5000)


In [41]:
y_train_flat = y_train.reshape(1, -1)
y_test_flat = y_test.reshape(1, -1)
print("y_train:", y_train_flat.shape)
print("y_test:", y_test_flat.shape)


y_train: (1, 20000)
y_test: (1, 5000)


In [ ]:

print(y_train.shape)

(1, 20000)


In [ ]:
res2 = fit(X_train_flat,X_test_flat,y_train_flat,y_test_flat,4,4,0.1,5,250)

100%|██████████| 80/80 [00:04<00:00, 17.27it/s]


   loss_train  accuracy_train  loss_test  accuracy_test
0    0.693163         0.50075   0.693219          0.497


100%|██████████| 80/80 [00:05<00:00, 14.75it/s]


   loss_train  accuracy_train  loss_test  accuracy_test
0    0.693163         0.50075   0.693219          0.497
1    0.693148         0.49925   0.693144          0.503


100%|██████████| 80/80 [00:17<00:00,  4.68it/s]


   loss_train  accuracy_train  loss_test  accuracy_test
0    0.693163         0.50075   0.693219          0.497
1    0.693148         0.49925   0.693144          0.503
2    0.693150         0.50075   0.693182          0.497


100%|██████████| 80/80 [00:05<00:00, 15.07it/s]


   loss_train  accuracy_train  loss_test  accuracy_test
0    0.693163         0.50075   0.693219          0.497
1    0.693148         0.49925   0.693144          0.503
2    0.693150         0.50075   0.693182          0.497
3    0.693152         0.49925   0.693137          0.503


100%|██████████| 80/80 [00:11<00:00,  6.79it/s]


   loss_train  accuracy_train  loss_test  accuracy_test
0    0.693163         0.50075   0.693219          0.497
1    0.693148         0.49925   0.693144          0.503
2    0.693150         0.50075   0.693182          0.497
3    0.693152         0.49925   0.693137          0.503
4    0.693150         0.49925   0.693140          0.503
   loss_train  accuracy_train  loss_test  accuracy_test
0    0.693163         0.50075   0.693219          0.497
1    0.693148         0.49925   0.693144          0.503
2    0.693150         0.50075   0.693182          0.497
3    0.693152         0.49925   0.693137          0.503
4    0.693150         0.49925   0.693140          0.503


In [21]:
## Je n'ai pas obtenu de grands résultats avec le réseaux de neurones connectés, j'utilise désormais les CNN
from tensorflow.keras import layers,models
model = models.Sequential()
## je cree ma premiere couche de convolution
## Je cree une couche de convoltion de 32 filtres, d'un kernel de dimension (3,3) pour des images de tailles 64x64x3 
## *3 represnte les canaux(RGB) et ma fonction d'activation relu, pour que le modele apprenne des relations complexes, pas linéaires
## Le padding représente la dimension de ma matrice de sortie, juste apres convoltion. En mettant padding="same" on concerve le max d'informations
model.add(layers.Conv2D(32,(3,3),activation="relu",padding='same',input_shape = (64,64,3)))

## Le MaxPooling utilise une fenetre de 2x2 pour résuire la dimension du résultat de la convolution
## Après le MaxPooling2D on aura une matrice de taille/2 c'est a dire 64/2=32 et de profondeur 32 filtres
model.add(layers.MaxPooling2D((2,2)))

model.add(layers.Conv2D(64,(3,3),activation='relu',padding='same'))
## La on applique 64 filtres et on force le resultat a etre de la même dimension que l'entrée
## On aura donc une matrice de taille 32 et de profondeur 64
## Chaque nouveau filtres utilise toutes les maps precedentes pour produires une seule map
model.add(layers.MaxPooling2D(2,2))
## La on a des matrices de tailles 32/2=16 et de profondeur 64

model.add(layers.Conv2D(128,(3,3),activation='relu',padding='same'))
model.add(layers.MaxPooling2D(2,2))
## ma taille est de 8x8 et de profondeur 128

model.add(layers.Conv2D(256,(3,3),activation='relu',padding='same'))
model.add(layers.MaxPooling2D(2,2))
## ma taille est de dimension 4x4 et de profondeur 256

## L'intéret de reduire la taille tout en augmentant les features est de diminuer au maximum les données d'entrées de mon futur réseau tout en conservant les caractéristiques(filtres) importants
## J'applaties maintenant mes données pour avoir un seul vecteur que je vais faire passer dans un réseau de neurones profonds
model.add(layers.Flatten())


## je cree mon reseau de neurones profonds
model.add(layers.Dense(128,activation='relu'))
model.add(layers.Dense(128,activation='relu'))
model.add(layers.Dense(1,"sigmoid"))


In [22]:
## L'entrainemet
model.compile(optimizer='adam',## optimizer represente l'algorithme utilisé pour mettre a jour les poids, par ex descente de gradient, ect
              loss = 'binary_crossentropy', ## la fonction qui calcule le score ou l'erreur
              metrics = ['accuracy'])## Permet a l'humain de voir l'exactitude de ces résultats

In [ ]:
history = model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=15,batch_size=32)

Epoch 1/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 48s 75ms/step - accuracy: 0.7085 - loss: 0.5619 - val_accuracy: 0.7766 - val_loss: 0.4858
Epoch 2/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 46s 74ms/step - accuracy: 0.7934 - loss: 0.4460 - val_accuracy: 0.8120 - val_loss: 0.4117
Epoch 3/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 46s 74ms/step - accuracy: 0.8332 - loss: 0.3753 - val_accuracy: 0.8294 - val_loss: 0.3759
Epoch 4/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 85s 78ms/step - accuracy: 0.8655 - loss: 0.3103 - val_accuracy: 0.8566 - val_loss: 0.3311
Epoch 5/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 51s 82ms/step - accuracy: 0.8858 - loss: 0.2705 - val_accuracy: 0.8636 - val_loss: 0.3396
Epoch 6/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 81s 130ms/step - accuracy: 0.9063 - loss: 0.2215 - val_accuracy: 0.8594 - val_loss: 0.3313
Epoch 7/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 50s 79ms/step - accuracy: 0.9284 - loss: 0.1794 - val_accuracy: 0.8662 - val_loss: 0.3588
Epoch 8/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 47s 76ms/step - accuracy: 0.9427 - loss: 0.1455 -